# 1-3절 연습 문제 풀이

이 노트북은 1-3절 연습 문제의 풀이 예시다. 정답이 하나뿐인 문제가 아니므로 다른 구현도 얼마든지 가능하다.

- 본문 예제 코드는 `notebooks/ch01/` 아래 예제 노트북을 참고한다.
- 위에서부터 차례대로 실행한다.

In [ ]:
# 환경 설정 - 공통 라이브러리, 시드 고정, 장치 객체
import sys
sys.path.append('../../')

import random

import numpy as np
import torch
import torch.nn as nn

from code_reference import common
# viz.configure()에서 save_grayscale=True로 지정하면 노트북에 표시되는 시각화 이미지를 파일로 저장함
from code_reference import visualize as viz

viz.configure(save_grayscale=False)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = common.get_device()

## 연습 1-9

파라미터의 값을 무작위로 초기화하는 대신, a의 초기값으로 0, 5, 10을, b의 초깃값으로 -3, 0, 3을 사용하는 아홉 개의 서로 다른 파라미터 초깃값으로 모델을 학습한 후 결과를 확인해 보자. 그 결과를 바탕으로 파라미터 초깃값과 모델 학습 결과를 설명해 보자.

In [ ]:
# 1-3절 예제 데이터(y = 4.9x^2 + 0 + 노이즈)를 다시 만든다.
def make_data(n_samples=50, seed=SEED):
    g = torch.Generator().manual_seed(seed)
    x = torch.rand(n_samples, 1, generator=g) * 10
    y = 4.9 * x ** 2
    y = y + torch.randn(n_samples, 1, generator=g) * (y * 0.1)
    return x, y

def train(a_init, b_init, x, y, epochs=2000, lr=1e-4):
    a = torch.tensor([[a_init]], requires_grad=True)
    b = torch.tensor([[b_init]], requires_grad=True)
    for _ in range(epochs):
        loss = ((a * x ** 2 + b - y) ** 2).mean()
        loss.backward()
        with torch.no_grad():
            a -= lr * a.grad
            b -= lr * b.grad
        a.grad.zero_(); b.grad.zero_()
    return a.item(), b.item(), loss.item()

x, y = make_data()
print(f'{"a 초깃값":>8} {"b 초깃값":>8} {"학습된 a":>10} {"학습된 b":>10} {"손실":>12}')
for a0 in (0.0, 5.0, 10.0):
    for b0 in (-3.0, 0.0, 3.0):
        a_hat, b_hat, loss = train(a0, b0, x, y)
        print(f'{a0:8.1f} {b0:8.1f} {a_hat:10.4f} {b_hat:10.4f} {loss:12.2f}')

아홉 가지 초깃값 모두 **a는 5.0 부근**(실제 4.9)으로 수렴한다. 손실 지형이 a에 대해 볼록해 어디서 출발하든 같은 최솟값으로 향하기 때문이다.

반면 **b는 초깃값 근처에 그대로 머문다**(-3.2, -0.6, 1.9 …). x² 항에 비해 b가 손실에 주는 영향이 훨씬 작아 기울기도 작고, 같은 학습률·에포크에서는 거의 갱신되지 않기 때문이다.

여기서 두 가지를 알 수 있다.

1. 초깃값은 **도달점보다 수렴 속도에 더 큰 영향**을 준다(a의 경우).
2. 파라미터마다 손실에 대한 민감도가 다르면, **덜 민감한 파라미터는 학습이 매우 느리다**(b의 경우). 에포크를 크게 늘리거나 파라미터별로 보폭을 조절하는 Adam 같은 옵티마이저를 쓰면 b도 0에 가까워진다.

## 연습 1-10

1-3절의 예제에서 샘플의 수를 충분히 늘려, 학습으로 구한 두 파라미터의 값이 4.9와 0에 가까워지는지 확인해 보자.

힌트: 샘플의 수에 따라 전체 학습 에포크도 수정해야 한다.

In [ ]:
# 샘플 수를 늘리면서 학습해 파라미터가 4.9와 0에 가까워지는지 확인한다.
print(f'{"샘플 수":>8} {"에포크":>8} {"학습된 a":>10} {"학습된 b":>10}')
for n_samples, epochs in [(50, 2000), (200, 4000), (1000, 8000), (5000, 20000)]:
    x, y = make_data(n_samples)
    a_hat, b_hat, _ = train(0.0, 0.0, x, y, epochs=epochs)
    print(f'{n_samples:8d} {epochs:8d} {a_hat:10.4f} {b_hat:10.4f}')

샘플이 많아질수록 노이즈가 서로 상쇄되어 a는 4.9에, b는 0에 더 가까워진다. 힌트대로 샘플 수를 늘리면 한 에포크에 반영되는 정보가 늘지만 수렴에 필요한 반복도 함께 늘어나므로 에포크도 함께 키웠다.

## 연습 1-11

1-3절의 예제의 손실 함수를 평균제곱오차 대신 평균절대오차를 사용해 손실을 계산하도록 수정한 후, 모델을 학습하고 그 결과를 비교해 보자.

In [ ]:
def train_with_loss(loss_name, x, y, epochs=4000, lr=1e-4):
    a = torch.tensor([[0.0]], requires_grad=True)
    b = torch.tensor([[0.0]], requires_grad=True)
    history = []
    for _ in range(epochs):
        pred = a * x ** 2 + b
        if loss_name == 'MSE':
            loss = ((pred - y) ** 2).mean()          # 평균제곱오차
        else:
            loss = (pred - y).abs().mean()           # 평균절대오차
        loss.backward()
        with torch.no_grad():
            a -= lr * a.grad
            b -= lr * b.grad
        a.grad.zero_(); b.grad.zero_()
        history.append(loss.item())
    return a.item(), b.item(), history

x, y = make_data(200)
for name in ('MSE', 'MAE'):
    a_hat, b_hat, hist = train_with_loss(name, x, y)
    print(f'{name}: a={a_hat:.4f}, b={b_hat:.4f}, 최종 손실={hist[-1]:.4f}')

평균절대오차(MAE)는 오차에 제곱을 하지 않으므로 큰 오차에 덜 민감하다. 그래서 이상치가 있을 때 더 안정적이지만, 오차가 작아져도 기울기 크기가 줄지 않아(부호만 남아) 최솟값 근처에서 수렴이 더디다.

같은 학습률에서 두 손실 함수를 비교하면 값의 규모 자체가 다르므로, 손실값의 크기보다 **파라미터가 정답에 얼마나 가까워졌는지**로 비교하는 것이 맞다.

## 연습 1-12

1-3절의 예제는 딥러닝으로 데이터를 y=ax2+b에 맞춰 a와 b의 값을 찾는 회귀 분석 모델이다. 그런데 이 데이터가 몇 차의 함수로 표현되는 모델에 적합한지 미리 알 수 없다면 지수까지 파라미터로 두는 모델을 사용할 수 있다. 회귀 분석 모델을 y=axc+b 로 바꾸고 a, b, c 세 파라미터의 값을 찾도록 예제를 수정해 보자.

In [ ]:
# y = a * x^c + b : 지수 c까지 파라미터로 둔다.
x, y = make_data(200)
# 값의 규모 차이가 크므로 x를 0~1로 줄여 학습을 안정시킨다.
x_scaled = x / 10.0
y_scaled = y / y.max()

a = torch.tensor([[1.0]], requires_grad=True)
b = torch.tensor([[0.0]], requires_grad=True)
c = torch.tensor([[1.0]], requires_grad=True)

optimizer = torch.optim.Adam([a, b, c], lr=0.01)
for epoch in range(3000):
    pred = a * x_scaled.clamp(min=1e-6) ** c + b     # 0의 거듭제곱을 피한다.
    loss = ((pred - y_scaled) ** 2).mean()
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    if (epoch + 1) % 1000 == 0:
        print(f'{epoch + 1:5d} 에포크 - 손실 {loss.item():.6f}, '
              f'a={a.item():.3f}, b={b.item():.3f}, c={c.item():.3f}')

print(f'\n찾은 지수 c = {c.item():.3f} (데이터를 만든 실제 지수는 2)')

지수를 파라미터로 두면 몇 차 함수인지 모르는 상태에서도 데이터에 맞는 차수를 찾을 수 있다. 다만 다음 두 가지에 주의해야 한다.

- `x ** c`는 x가 0이거나 음수일 때 정의되지 않거나 기울기가 불안정하므로 `clamp()`로 막았다.
- 지수 연산은 값의 규모가 급격히 커져 발산하기 쉽다. 여기서는 입력과 정답을 0~1 범위로 줄이고, 학습률을 자동으로 조절하는 Adam 옵티마이저를 사용했다.

## 연습 1-13

[도전 문제] 학습 곡선을 보면 학습 초반에는 손실이 크고, 학습이 진행되면서 손실이 감소한다. '손실이 클 때는 강하게, 손실이 작을 때는 약하게 학습한다.'는 명제는 딥러닝에만 국한되지 않는 일반적인 원리다. 1-3절의 예제에 이 아이디어를 적용해 보자.

1장 학습 노트

In [ ]:
# 손실이 클 때는 크게, 작을 때는 작게 - 손실에 비례해 학습률을 조절한다.
x, y = make_data(200)
x_s, y_s = x / 10.0, y / y.max()

def train_adaptive(mode, epochs=3000, base_lr=0.05):
    a = torch.tensor([[0.0]], requires_grad=True)
    b = torch.tensor([[0.0]], requires_grad=True)
    hist = []
    for _ in range(epochs):
        loss = ((a * x_s ** 2 + b - y_s) ** 2).mean()
        loss.backward()
        if mode == '고정':
            lr = base_lr * 0.1
        elif mode == '손실 비례':
            lr = base_lr * min(loss.item(), 1.0)
        else:   # 하한을 둔 손실 비례
            lr = base_lr * max(min(loss.item(), 1.0), 0.1)
        with torch.no_grad():
            a -= lr * a.grad
            b -= lr * b.grad
        a.grad.zero_(); b.grad.zero_()
        hist.append(loss.item())
    return hist

results = {m: train_adaptive(m) for m in ('고정', '손실 비례', '하한 있는 손실 비례')}
for m, h in results.items():
    print(f'{m:16s} 100 에포크 {h[99]:.5f} / 최종 {h[-1]:.6f}')

viz.plot_histories(results, title='학습률 조절 방식에 따른 학습 곡선', y_label='손실')

**실행 결과를 보면 아이디어가 그대로 통하지는 않는다.**

- **초반**에는 손실 비례 방식이 큰 보폭으로 빠르게 내려간다(100 에포크 시점 비교).
- **후반**에는 손실이 작아지면서 학습률까지 함께 작아져 **거의 멈춰 버린다**. 그래서 최종 손실은 고정 학습률보다 오히려 나쁘다.

학습률에 **하한을 두면**(세 번째 방식) 초반의 빠른 수렴을 살리면서 후반 정체도 피할 수 있다.

실제 딥러닝에서 이 아이디어를 구현한 것이 **학습률 스케줄러**(`torch.optim.lr_scheduler`)와 **적응형 옵티마이저**(Adam)다. 다만 둘 다 손실값에 직접 비례시키지 않고, 기울기의 이력이나 정해진 일정에 따라 보폭을 조절한다. 이 실험이 그 이유를 보여 준다.